### Step 1: Import Libraries & API Keys

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
import gradio as gr
import json
import requests

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing.")

client = OpenAI(api_key=OPENAI_API_KEY)

### Step 2: Set up Pushover

In [2]:
# Step 2a -> Setu up account in your browser
# Step 2b -> Set up the app on your phone
# Step 2c -> In the browser create an "Application/API Token"
# Step 2d -> Add the User Key and API token to your .env file
# Step 2e -> Install the Pushover app on your phone and log in with the same account
# Step 2f -> Run the code below to send a test notification to your phone

load_dotenv()

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"


### Step 3: Test Pushover

In [3]:
def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

### Step 4: Describe Pushover as an LLM tool

In [5]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a notification to the user's phone using the Pushover service.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The message to send in the notification."
            }
        },
        "required": ["message"]
    }
}

### Step 5: Add Pushover to the list of tools for the LLM

In [6]:
tools = [{"type": "function", "function": send_notification_function}]

### Step 6: Calling the tool from an LLM

In [13]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)

        print(f"Handling tool call for function: {function_name} with arguments: {args}") # For debugging

        # Route to the appropriate function based on function_name
        if function_name == "send_notification":
            # Actually send the notification, i.e. call the tool
            send_notification(args["message"])
            content = f"Notification sent: {args['message']}"
        else:
            content = f"Unknown tool call: {function_name}"

        tool_results.append({
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        })

    # Return what to add to the context about tool call results, a list of dictionaries
    return tool_results


In [14]:
client = OpenAI(api_key=OPENAI_API_KEY)

messages=[
    {"role": "user", "content": "Please send me two separate notifications: 1) I'm making great progress on the AI Engineering course! 2) I just completed a challenging project!"}
]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)

message = response.choices[0].message

# Check if model wants to call a tool
if message.tool_calls:
    # Handle the tool call
    tool_result = handle_tool_call(message.tool_calls) # Whole list of tool calls
    # Add message to context, i.e. messages
    messages.append(message)
    # Add Info about tool call response to the message content
    messages.extend(tool_result) # Changed append() to extend() when switched to handling multiple tool calls
    # Invoke the LLM one more time to get its update response
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )
    message = response.choices[0].message

print(message.content)

Handling tool call for function: send_notification with arguments: {'message': "I'm making great progress on the AI Engineering course!"}
Handling tool call for function: send_notification with arguments: {'message': 'I just completed a challenging project!'}
Both notifications have been sent:

1) I'm making great progress on the AI Engineering course!
2) I just completed a challenging project!
